In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

In [2]:
import tarfile

tar_path = r"caltech101\caltech-101\101_ObjectCategories.tar.gz"

extract_path = r"caltech101\caltech-101"

dataset_path = os.path.join(extract_path, "101_ObjectCategories")

if not os.path.exists(dataset_path):

    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_path)

    print("Dataset extracted successfully!")

else:
    print("Dataset already extracted!")

Dataset already extracted!


In [3]:
class_names = [
    'airplanes',
    'Faces',
    'Motorbikes',
    'Leopards',
    'watch'
]

dataset_path = r"caltech101\caltech-101\101_ObjectCategories"

for category in class_names:

    category_path = os.path.join(dataset_path, category)

    print(category, ":", os.path.isdir(category_path))

airplanes : True
Faces : True
Motorbikes : True
Leopards : True
watch : True


In [4]:
image_paths = []
labels = []

for label, category in enumerate(class_names):

    folder = os.path.join(dataset_path, category)

    for filename in os.listdir(folder):

        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):

            image_paths.append(os.path.join(folder, filename))
            labels.append(label)

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Total images:", len(image_paths))
print("Total labels:", len(labels))
print("Categories:", class_names)

Total images: 2472
Total labels: 2472
Categories: ['airplanes', 'Faces', 'Motorbikes', 'Leopards', 'watch']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))

Training images: 1977
Testing images: 495


In [6]:
IMG_SIZE = (224, 224)

def load_image(path):

    image = tf.io.read_file(path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(image, IMG_SIZE)

    image = preprocess_input(image)

    return image


def create_dataset(paths, labels):

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        lambda path, label: (
            load_image(path),
            label
        ),
        num_parallel_calls=1
    )

    dataset = dataset.batch(8).prefetch(1)

    return dataset


train_ds = create_dataset(X_train, y_train)
test_ds = create_dataset(X_test, y_test)

print("Image preprocessing completed!")

Image preprocessing completed!


In [7]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all VGG16 layers
base_model.trainable = False

print("VGG16 loaded successfully!")

VGG16 loaded successfully!


In [8]:
inputs = tf.keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs, x)

print("Feature extractor created successfully!")

Feature extractor created successfully!


In [ ]:
def extract_features(dataset):

    features = []
    labels_list = []

    for images, labels_batch in dataset:

        batch_features = feature_extractor.predict(
            images,
            verbose=0
        )

        features.append(batch_features)
        labels_list.append(labels_batch.numpy())

    features = np.concatenate(features)
    labels_list = np.concatenate(labels_list)

    return features, labels_list


X_train_features, y_train_features = extract_features(train_ds)

X_test_features, y_test_features = extract_features(test_ds)

print("Training feature shape:", X_train_features.shape)
print("Testing feature shape:", X_test_features.shape)

In [ ]:
num_classes = len(class_names)

classifier = tf.keras.Sequential([
    Dense(256, activation='relu',
          input_shape=(X_train_features.shape[1],)),

    Dense(num_classes, activation='softmax')
])

classifier.summary()

In [ ]:
classifier.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Classifier compiled successfully!")

In [ ]:
history = classifier.fit(
    X_train_features,
    y_train_features,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

In [ ]:
test_loss, test_accuracy = classifier.evaluate(
    X_test_features,
    y_test_features
)

print("Test Loss:", test_loss)

print("Classification Accuracy:", test_accuracy * 100, "%")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.xlabel('Epochs')
plt.ylabel('Accuracy')

plt.title('VGG16 Feature Extraction Accuracy')

plt.legend()
plt.show()

In [ ]:
predictions = classifier.predict(
    X_test_features[:5]
)

predicted_labels = np.argmax(predictions, axis=1)

plt.figure(figsize=(15, 6))

for i in range(5):

    image = load_image(X_test[i]).numpy()

    # Reverse VGG16 preprocessing for display
    image = image + [103.939, 116.779, 123.68]
    image = image[..., ::-1]

    image = np.clip(image, 0, 255).astype('uint8')

    plt.subplot(1, 5, i + 1)

    plt.imshow(image)

    plt.title(
        "Predicted: " + class_names[predicted_labels[i]] +
        "\nActual: " + class_names[y_test[i]]
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 26. Implement an object classification system using VGG16 as a feature
# extractor and the Caltech-101 dataset. Perform the following steps:
# a. Load the Caltech-101 dataset and select suitable object categories.
# b. Resize all images to 224 × 224 pixels and apply appropriate preprocessing.
# c. Load the ImageNet pre-trained VGG16 model without the top classification layers.
# d. Freeze all convolutional layers of VGG16 to use the network as a fixed feature
# extractor.
# e. Extract image features using the pre-trained convolutional layers and
# GlobalAveragePooling2D.
# f. Add a custom Dense classification layer with Softmax activation to classify the
# selected object categories.
# g. Train the custom classifier using the extracted training features.
# h. Evaluate the classifier using test features and display the classification accuracy